# Eval Viewer
Interactive notebook to visualize per-video predictions from a trained model.

1. Set `RESULTS_DIR` to a completed training run (the folder containing `config.yml` and fold subfolders).
2. Run all cells.
3. Use the slider to browse videos and see segment predictions + probability curves.

In [ ]:
# ── Config ──────────────────────────────────────────────
RESULTS_DIR = "/code/jjiang23/BalanceTestThesis/results/MAMP/MB/downsamp/ASFormer/20260603_224342"
DEVICE = "cuda"        # or "cpu"
STRIDE_OVERRIDE = 15   # smaller stride = smoother stitching (None = use data config stride)

In [ ]:
import os, sys, json, yaml
import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(RESULTS_DIR), "..", "..", "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.eval.metric_utils import predict_video, compute_segmentation_metrics, extract_segments

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ── Load config ─────────────────────────────────────────
config_path = os.path.join(RESULTS_DIR, "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

e_cfg = config["encoder"]
d_cfg = config["data"]
t_cfg = config["trainer"]
s_cfg = config["segmentor"]
splits_path = config["paths"]["splits_path"]

with open(splits_path) as f:
    splits = json.load(f)

print(f"Loaded config from: {config_path}")
print(f"Folds: {list(splits.keys())}")

In [ ]:
# ── Dynamic import helper ───────────────────────────────
def load_module_from_path(file_path):
    spec = importlib.util.spec_from_file_location("mod", file_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# ── Load initializers ───────────────────────────────────
if e_cfg is not None:
    init_encoder_path = os.path.abspath(e_cfg["init_encoder_path"])
else:
    init_encoder_path = os.path.join(PROJECT_ROOT, "initializers", "encoder", "Identity.py")

init_segmentor_path = os.path.abspath(s_cfg["init_segmentor_path"])

initialize_encoder = getattr(load_module_from_path(init_encoder_path), "initialize_encoder")
initialize_segmentor = getattr(load_module_from_path(init_segmentor_path), "initialize_segmentor")

# ── Build models (architecture only — weights loaded per fold) ──
encoder = initialize_encoder(d_cfg, e_cfg)
segmentor = initialize_segmentor(
    s_cfg, encoder,
    class_weights=None,
    lambda_smooth=t_cfg.get("lambda_smooth", 0.01),
    time_alignment=t_cfg.get("time_alignment", "downsample_labels"),
)
encoder.to(DEVICE).eval()
segmentor.to(DEVICE).eval()
print("Models ready.")

In [ ]:
# ── Collect all (fold, video) pairs ─────────────────────
video_entries = []  # list of (fold_name, video_path, fold_dir)

for fold_name, split_files in splits.items():
    fold_dir = os.path.join(RESULTS_DIR, fold_name)
    enc_ckpt = os.path.join(fold_dir, "best_encoder.pt")
    seg_ckpt = os.path.join(fold_dir, "best_segmentor.pt")
    if not (os.path.exists(enc_ckpt) and os.path.exists(seg_ckpt)):
        print(f"Skipping {fold_name}: no checkpoints")
        continue
    for vid_path in split_files["val"]:
        video_entries.append((fold_name, vid_path, fold_dir))

print(f"Total eval videos: {len(video_entries)} across {len(splits)} folds")

In [ ]:
# ── Cache for loaded fold weights ──────────────────────
_loaded_fold = None

def load_fold_weights(fold_dir):
    """Load encoder + segmentor weights for a fold (cached)."""
    global _loaded_fold
    if _loaded_fold == fold_dir:
        return
    encoder.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_encoder.pt"), map_location=DEVICE)
    )
    segmentor.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_segmentor.pt"), map_location=DEVICE),
        strict=False,
    )
    encoder.eval()
    segmentor.eval()
    _loaded_fold = fold_dir

In [ ]:
# ── Video frame utilities ──────────────────────────
import h5py
import cv2

def get_video_meta_from_h5(h5_path):
    """
    Returns (video_path, box_coords) from h5 file.
    box_coords: (x1, y1, x2, y2) in pixel space, or None if not present.
    """
    try:
        with h5py.File(h5_path, 'r') as f:
            vp = f.attrs.get('video_path', None)
            if vp is not None:
                vp = vp.decode() if isinstance(vp, bytes) else str(vp)
            box = None
            if 'static_box_coords' in f:
                box = f['static_box_coords'][:].tolist()
            print(f"Read meta from {h5_path}: video_path={vp}, box_coords={box}")
        return vp, box
    except Exception as e:
        print(f'Warning: could not read meta from {h5_path}: {e}')
        return None, None


def read_video_frame(video_path, frame_idx):
    """Read a single RGB frame from a video file using OpenCV."""
    if video_path is None:
        return None
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f'Cannot open video: {video_path}')
            return None
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            return None
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f'Warning: frame {frame_idx} read failed: {e}')
        return None


print('Video frame utilities ready.')

In [ ]:
# ── Visualization helpers + prediction cache ─────────────
import matplotlib.patches as mpatches

CLASS_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107', '#9C27B0']
CLASS_NAMES  = ['Phase', 'Nonphase']

# ── Global prediction cache ────────────────────────────────────────────────
_pred_cache = {
    'vid_idx': None, 'gt': None, 'pred': None, 'probs': None,
    'video_path': None, 'box_coords': None,
    'fold_name': None, 'vid_name': None,
    'metrics': None, 'frame_acc': None,
    'T': 0,           # total frame count (set in both modes)
    'video_only': False,  # True when loaded without predictions
}


def _ensure_video_meta(vid_idx):
    """Load only h5 metadata (video_path, box, frame count) — no inference."""
    if _pred_cache['vid_idx'] == vid_idx and _pred_cache['video_only']:
        return
    fold_name, vid_path, _ = video_entries[vid_idx]
    video_path, box_coords = get_video_meta_from_h5(vid_path)
    # Get frame count from h5
    T = 0
    try:
        with h5py.File(vid_path, 'r') as f:
            T = len(f['camera_poses_labels'][:])
    except Exception:
        pass
    _pred_cache.update({
        'vid_idx':    vid_idx,
        'gt':         None,
        'pred':       None,
        'probs':      None,
        'video_path': video_path,
        'box_coords': box_coords,
        'fold_name':  fold_name,
        'vid_name':   os.path.basename(vid_path),
        'metrics':    None,
        'frame_acc':  None,
        'T':          T,
        'video_only': True,
    })


def _ensure_predictions(vid_idx):
    """Run inference for vid_idx — skipped if already cached with predictions."""
    if _pred_cache['vid_idx'] == vid_idx and not _pred_cache['video_only']:
        return
    fold_name, vid_path, fold_dir = video_entries[vid_idx]
    load_fold_weights(fold_dir)
    gt, pred, probs = predict_video(
        vid_path, encoder, segmentor, d_cfg, DEVICE,
        stride_override=STRIDE_OVERRIDE,
    )
    valid = gt != -100
    metrics = compute_segmentation_metrics(
        gt, pred, class_id=0, iou_thresholds=(0.1, 0.25, 0.5), fps=30,
    )
    video_path, box_coords = get_video_meta_from_h5(vid_path)
    _pred_cache.update({
        'vid_idx':    vid_idx,
        'gt':         gt,
        'pred':       pred,
        'probs':      probs,
        'video_path': video_path,
        'box_coords': box_coords,
        'fold_name':  fold_name,
        'vid_name':   os.path.basename(vid_path),
        'metrics':    metrics,
        'frame_acc':  float((gt[valid] == pred[valid]).mean()) if valid.sum() > 0 else 0.0,
        'T':          len(gt),
        'video_only': False,
    })


def _draw_chart(frame_cursor=None):
    """Draw 3-row GT/Pred/Prob chart with optional cursor line."""
    gt    = _pred_cache['gt']
    pred  = _pred_cache['pred']
    probs = _pred_cache['probs']
    T = len(gt)
    frames = np.arange(T)
    num_classes = probs.shape[1]
    m = _pred_cache['metrics']
    f1_10  = m.get('f1_iou_0.1',  0) or 0
    f1_25  = m.get('f1_iou_0.25', 0) or 0
    f1_50  = m.get('f1_iou_0.5',  0) or 0

    fig, axes = plt.subplots(
        3, 1, figsize=(18, 7), sharex=True,
        gridspec_kw={'height_ratios': [1, 1, 2.5], 'hspace': 0.08},
    )
    ax_gt, ax_pred, ax_prob = axes

    for c in range(num_classes):
        color = CLASS_COLORS[c % len(CLASS_COLORS)]
        name  = CLASS_NAMES[c] if c < len(CLASS_NAMES) else f'Class {c}'
        ax_gt.fill_between(frames, 0, 1, where=(gt == c),
                            color=color, alpha=0.9, label=name)
        ax_pred.fill_between(frames, 0, 1, where=(pred == c),
                              color=color, alpha=0.9)
        ax_prob.plot(frames, probs[:, c], label=name, color=color, lw=1.2)

    ax_gt.set_yticks([]);   ax_gt.set_ylabel('GT',   fontsize=10, fontweight='bold')
    ax_pred.set_yticks([]); ax_pred.set_ylabel('Pred', fontsize=10, fontweight='bold')
    ax_gt.legend(loc='upper right', fontsize=7, ncol=num_classes)
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.set_ylabel('Probability', fontsize=10)
    ax_prob.set_xlabel('Frame', fontsize=10)
    ax_prob.legend(loc='upper right', fontsize=7)
    ax_prob.grid(axis='y', alpha=0.3)

    if frame_cursor is not None:
        for ax in axes:
            ax.axvline(x=frame_cursor, color='white', lw=1.8, alpha=0.9, zorder=5)

    vid_name  = _pred_cache['vid_name']
    fold_name = _pred_cache['fold_name']
    frame_acc = _pred_cache['frame_acc']
    fig.suptitle(
        f'[{fold_name}] {vid_name}   |   Acc: {frame_acc:.1%}   '
        f'F1@.10: {f1_10:.2f}  F1@.25: {f1_25:.2f}  F1@.50: {f1_50:.2f}   ({T} frames)',
        fontsize=11, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    plt.show()


def _draw_frame(frame_idx):
    """Display the video frame.  GT/pred labels shown only when predictions are cached."""
    vp  = _pred_cache['video_path']
    box = _pred_cache['box_coords']
    gt  = _pred_cache['gt']
    pred = _pred_cache['pred']

    fig, ax = plt.subplots(figsize=(6, 5))
    frame_img = read_video_frame(vp, frame_idx) if vp else None

    if frame_img is not None:
        ax.imshow(frame_img)

        # Draw static bounding box
        if box is not None:
            box_flat = np.array(box).flatten()
            if len(box_flat) == 4:
                x1, y1, x2, y2 = box_flat
                rect = mpatches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=3, edgecolor='lime', facecolor='none', zorder=10,
                    label=f'box ({x1:.0f},{y1:.0f})\u2192({x2:.0f},{y2:.0f})',
                )
                ax.add_patch(rect)
                ax.legend(loc='lower right', fontsize=7, framealpha=0.7)

        if gt is not None and pred is not None and frame_idx < len(gt):
            gt_lbl   = int(gt[frame_idx])
            pred_lbl = int(pred[frame_idx])
            gt_name   = CLASS_NAMES[gt_lbl]   if 0 <= gt_lbl   < len(CLASS_NAMES) else str(gt_lbl)
            pred_name = CLASS_NAMES[pred_lbl] if 0 <= pred_lbl < len(CLASS_NAMES) else str(pred_lbl)
            correct   = '\u2713' if gt_lbl == pred_lbl else '\u2717'
            color     = 'green' if gt_lbl == pred_lbl else 'red'
            ax.set_title(
                f'Frame {frame_idx}   GT: {gt_name}   Pred: {pred_name}   {correct}',
                fontsize=9, color=color, fontweight='bold',
            )
        else:
            ax.set_title(f'Frame {frame_idx}  [{_pred_cache["vid_name"]}]', fontsize=9)
    else:
        msg = (f'Frame {frame_idx}'
               + ('\n(video not found)' if vp else '\n(no video_path in h5 attrs)'))
        ax.text(0.5, 0.5, msg, ha='center', va='center',
                transform=ax.transAxes, fontsize=11)
        ax.set_title(f'Frame {frame_idx}')

    ax.axis('off')
    plt.tight_layout()
    plt.show()


print('Visualization helpers ready.')

In [ ]:
# ── Interactive viewer ────────────────────────────────
from ipywidgets import Output, VBox, HBox, Dropdown, IntSlider, Checkbox
from IPython.display import display, clear_output

chart_out = Output()
frame_out = Output()

# Build display labels: "[fold] filename"
video_options = {
    f'[{fold}] {os.path.basename(vid)}': i
    for i, (fold, vid, _) in enumerate(video_entries)
}

video_dd = Dropdown(
    options=video_options,
    value=0,
    description='Video:',
    style={'description_width': '60px'},
    layout={'width': '640px'},
)

frame_slider = IntSlider(
    min=0, max=0, value=0,
    description='Frame:',
    style={'description_width': '60px'},
    layout={'width': '700px'},
    continuous_update=False,
)

video_only_toggle = Checkbox(
    value=False,
    description='Video only (skip predictions)',
    indent=False,
    layout={'width': '260px'},
)


def _refresh(vid_idx, frame_idx):
    with frame_out:
        clear_output(wait=True)
        _draw_frame(frame_idx)
    with chart_out:
        clear_output(wait=True)
        if video_only_toggle.value:
            # Show lightweight placeholder instead of running inference
            fig, ax = plt.subplots(figsize=(18, 2))
            ax.text(0.5, 0.5,
                    f'Predictions disabled  —  "{_pred_cache["vid_name"]}"  '
                    f'({_pred_cache["T"]} frames)\n'
                    'Uncheck "Video only" to compute predictions.',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=12, color='gray')
            ax.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            _draw_chart(frame_cursor=frame_idx)


def on_video_change(change):
    vid_idx = change['new']
    if video_only_toggle.value:
        _ensure_video_meta(vid_idx)
    else:
        _ensure_predictions(vid_idx)
    T = _pred_cache['T']
    frame_slider.unobserve(on_frame_change, names='value')
    frame_slider.max = max(T - 1, 0)
    frame_slider.value = 0
    frame_slider.observe(on_frame_change, names='value')
    _refresh(vid_idx, 0)


def on_frame_change(change):
    if _pred_cache['vid_idx'] is None:
        return
    _refresh(_pred_cache['vid_idx'], change['new'])


def on_toggle_change(change):
    """When toggled, reload current video in the appropriate mode."""
    vid_idx = video_dd.value
    if change['new']:   # just enabled video-only
        _ensure_video_meta(vid_idx)
    else:               # just disabled — compute predictions now
        _ensure_predictions(vid_idx)
        T = _pred_cache['T']
        frame_slider.unobserve(on_frame_change, names='value')
        frame_slider.max = max(T - 1, 0)
        frame_slider.observe(on_frame_change, names='value')
    _refresh(vid_idx, frame_slider.value)


video_dd.observe(on_video_change, names='value')
frame_slider.observe(on_frame_change, names='value')
video_only_toggle.observe(on_toggle_change, names='value')

# Bootstrap: respect the current toggle state
if video_only_toggle.value:
    _ensure_video_meta(0)
else:
    _ensure_predictions(0)
frame_slider.max = max(_pred_cache['T'] - 1, 0)
_refresh(0, 0)

display(VBox([
    HBox([video_dd, video_only_toggle]),
    frame_slider,
    frame_out,
    chart_out,
]))